# Мини-проект: A/A/B-тестирование маркетплейса

Цель работы: проверить качество данных, рассчитать ключевые метрики маркетплейса, проверить корректность A/A-сплитов `A` и `C`, а затем оценить группу `B` относительно `A`.

Группы:
- `sample_a`, `sample_c` — A/A-группы;
- `sample_b` — тестовая группа B.

Действия пользователей:
- `0` — клик;
- `1` — просмотр;
- `2` — покупка.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

ALPHA = 0.01
ALPHA_SHAPIRO = 0.01

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.6g}")

DATA_DIR = Path("data")
assert DATA_DIR.exists(), "Папка data не найдена. Положите её рядом с ноутбуком."

sample_a = pd.read_csv(DATA_DIR / "sample_a.csv")
sample_b = pd.read_csv(DATA_DIR / "sample_b.csv")
sample_c = pd.read_csv(DATA_DIR / "sample_c.csv")
item_prices = pd.read_csv(DATA_DIR / "item_prices.csv")

groups_raw = {"A": sample_a, "B": sample_b, "C": sample_c}

print("Данные успешно загружены")
for name, df in groups_raw.items():
    print(f"sample_{name.lower()}: {df.shape[0]:,} строк, {df.shape[1]} столбца")
print(f"item_prices: {item_prices.shape[0]:,} строк, {item_prices.shape[1]} столбца")

Данные успешно загружены
sample_a: 1,188,912 строк, 3 столбца
sample_b: 1,198,438 строк, 3 столбца
sample_c: 1,205,510 строк, 3 столбца
item_prices: 1,000 строк, 2 столбца


## 1. Проверка качества данных

Требования:
1. удалить дубли;
2. проверить, что нет ситуаций, когда у пары `user_id + item_id` есть клик или покупка, но нет просмотра.

Проверку делаем на уровне пары `user_id + item_id`, потому что действие просмотра должно предшествовать клику/покупке по конкретному товару.

In [2]:
def clean_and_check(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Удаляет дубли и проверяет невозможные цепочки действий."""
    required_columns = {"user_id", "item_id", "action_id"}
    missing_columns = required_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"В датасете не хватает колонок: {missing_columns}")

    raw_rows = len(df)
    duplicate_rows = int(df.duplicated().sum())
    unexpected_actions = sorted(set(df["action_id"].unique()) - {0, 1, 2})

    df = df.drop_duplicates().copy()

    action_matrix = (
        df.assign(cnt=1)
          .pivot_table(
              index=["user_id", "item_id"],
              columns="action_id",
              values="cnt",
              aggfunc="sum",
              fill_value=0,
          )
    )

    for action_id in [0, 1, 2]:
        if action_id not in action_matrix.columns:
            action_matrix[action_id] = 0

    click_without_view = (action_matrix[0] > 0) & (action_matrix[1] == 0)
    purchase_without_view = (action_matrix[2] > 0) & (action_matrix[1] == 0)
    invalid_any = click_without_view | purchase_without_view

    rows_removed_as_invalid = 0
    if invalid_any.any():
        bad_pairs = pd.MultiIndex.from_tuples(
            list(action_matrix.index[invalid_any]),
            names=["user_id", "item_id"],
        )
        current_pairs = pd.MultiIndex.from_frame(df[["user_id", "item_id"]])
        bad_row_mask = current_pairs.isin(bad_pairs)
        rows_removed_as_invalid = int(bad_row_mask.sum())
        df = df.loc[~bad_row_mask].copy()

    check_info = {
        "raw_rows": raw_rows,
        "duplicate_rows": duplicate_rows,
        "rows_after_dedup": raw_rows - duplicate_rows,
        "click_without_view_pairs": int(click_without_view.sum()),
        "purchase_without_view_pairs": int(purchase_without_view.sum()),
        "invalid_any_pairs": int(invalid_any.sum()),
        "rows_removed_as_invalid": rows_removed_as_invalid,
        "final_rows": len(df),
        "unexpected_actions": unexpected_actions,
    }

    return df, check_info

cleaned = {}
quality = {}

for group_name, df in groups_raw.items():
    cleaned[group_name], quality[group_name] = clean_and_check(df)

quality_df = pd.DataFrame(quality).T
quality_df

,raw_rows,duplicate_rows,rows_after_dedup,click_without_view_pairs,purchase_without_view_pairs,invalid_any_pairs,rows_removed_as_invalid,final_rows,unexpected_actions
A,1188912,0,1188912,0,0,0,0,1188912,[]
B,1198438,0,1198438,0,0,0,0,1198438,[]
C,1205510,0,1205510,0,0,0,0,1205510,[]


**Вывод по качеству данных:** дублей нет, клик/покупка без просмотра не обнаружены. Данные можно использовать для расчёта метрик.


## 2. Расчёт метрик

In [3]:
def calculate_metrics(df: pd.DataFrame, prices: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    counts = df["action_id"].value_counts()

    views = int(counts.get(1, 0))
    clicks = int(counts.get(0, 0))
    purchases = int(counts.get(2, 0))
    users = int(df["user_id"].nunique())

    purchase_df = df[df["action_id"] == 2].merge(prices, on="item_id", how="left")
    missing_prices = int(purchase_df["item_price"].isna().sum())
    if missing_prices > 0:
        raise ValueError(f"Для {missing_prices} покупок не найдены цены в item_prices.")

    gmv_total = float(purchase_df["item_price"].sum())

    user_actions = df.pivot_table(
        index="user_id",
        columns="action_id",
        values="item_id",
        aggfunc="count",
        fill_value=0,
    )

    for action_id in [0, 1, 2]:
        if action_id not in user_actions.columns:
            user_actions[action_id] = 0

    user_actions = user_actions.rename(columns={0: "clicks", 1: "views", 2: "purchases"})
    user_actions = user_actions[["clicks", "views", "purchases"]]

    user_gmv = purchase_df.groupby("user_id")["item_price"].sum()
    user_actions["gmv"] = user_actions.index.map(user_gmv).fillna(0).astype(float)
    user_actions["ctr"] = user_actions["clicks"] / user_actions["views"].replace(0, np.nan)
    user_actions["purchase_rate"] = user_actions["purchases"] / user_actions["views"].replace(0, np.nan)

    metrics = {
        "users": users,
        "views": views,
        "clicks": clicks,
        "purchases": purchases,
        "ctr": clicks / views,
        "purchase_rate": purchases / views,
        "gmv_total": gmv_total,
        "gmv_per_user": gmv_total / users,
        "gmv_per_view": gmv_total / views,
    }

    return metrics, user_actions.reset_index()

metrics = {}
user_metrics = {}

for group_name, df in cleaned.items():
    metrics[group_name], user_metrics[group_name] = calculate_metrics(df, item_prices)

metrics_df = pd.DataFrame(metrics).T
metrics_display = metrics_df.copy()
metrics_display["ctr_%"] = metrics_display["ctr"] * 100
metrics_display["purchase_rate_%"] = metrics_display["purchase_rate"] * 100

metrics_display[[
    "users", "views", "clicks", "purchases",
    "ctr_%", "purchase_rate_%",
    "gmv_total", "gmv_per_user", "gmv_per_view",
]]

,users,views,clicks,purchases,ctr_%,purchase_rate_%,gmv_total,gmv_per_user,gmv_per_view
A,996,"951,130","190,226","47,556",20,4.99995,5.30602e+07,"53,273.3",55.7865
B,996,"951,141","152,183","95,114",16,9.99999,1.06047e+08,"106,473",111.495
C,994,"949,221","199,336","56,953",21,5.99997,6.36149e+07,"63,998.9",67.018


### Общее сравнение метрик

По сырым метрикам видно:

- группа `C` отличается от `A`, хотя обе группы должны быть A/A-контролями;
- в `B` ниже CTR, но заметно выше purchase rate и GMV;
- из-за различий между `A` и `C` итоговый вывод по `B` нужно делать осторожно: сначала проверяем, не «разъехались» ли сплиты.


In [4]:
comparison_vs_a = metrics_df.loc[["B", "C"], ["ctr", "purchase_rate", "gmv_per_user", "gmv_per_view"]].copy()
comparison_vs_a = comparison_vs_a.subtract(metrics_df.loc["A", ["ctr", "purchase_rate", "gmv_per_user", "gmv_per_view"]])
comparison_vs_a.index = ["B - A", "C - A"]
comparison_vs_a["ctr_diff_pp"] = comparison_vs_a["ctr"] * 100
comparison_vs_a["purchase_rate_diff_pp"] = comparison_vs_a["purchase_rate"] * 100
comparison_vs_a[["ctr_diff_pp", "purchase_rate_diff_pp", "gmv_per_user", "gmv_per_view"]]

,ctr_diff_pp,purchase_rate_diff_pp,gmv_per_user,gmv_per_view
B - A,-3.99995,5.00004,"53,200",55.7084
C - A,0.999957,1.00003,"10,725.6",11.2316


## 3. Тест равенства долей для CTR и Purchase rate

`CTR` и `purchase rate` — это доли, поэтому для них используем z-test равенства долей.


In [5]:
def format_p_value(p_value: float) -> str:
    if p_value == 0:
        return "< 1e-300"
    return f"{p_value:.3e}"


def proportion_test(group_1: str, group_2: str, numerator_key: str, metric_name: str) -> dict:
    count = np.array([metrics[group_1][numerator_key], metrics[group_2][numerator_key]], dtype=float)
    nobs = np.array([metrics[group_1]["views"], metrics[group_2]["views"]], dtype=float)

    z_stat, p_value = proportions_ztest(count, nobs, alternative="two-sided")

    rate_1 = count[0] / nobs[0]
    rate_2 = count[1] / nobs[1]

    return {
        "comparison": f"{group_1} vs {group_2}",
        "metric": metric_name,
        "group_1": group_1,
        "group_2": group_2,
        "group_1_rate": rate_1,
        "group_2_rate": rate_2,
        "diff_group2_minus_group1": rate_2 - rate_1,
        "diff_pp": (rate_2 - rate_1) * 100,
        "z_stat": z_stat,
        "p_value": p_value,
        "p_value_fmt": format_p_value(p_value),
        "reject_h0_alpha_0.01": p_value < ALPHA,
    }

prop_tests = []
for group_1, group_2 in [("A", "C"), ("A", "B")]:
    prop_tests.append(proportion_test(group_1, group_2, "clicks", "CTR"))
    prop_tests.append(proportion_test(group_1, group_2, "purchases", "Purchase rate"))

prop_tests_df = pd.DataFrame(prop_tests)
prop_tests_df[[
    "comparison", "metric", "group_1_rate", "group_2_rate", "diff_pp",
    "z_stat", "p_value_fmt", "reject_h0_alpha_0.01",
]]

,comparison,metric,group_1_rate,group_2_rate,diff_pp,z_stat,p_value_fmt,reject_h0_alpha_0.01
0,A vs C,CTR,0.2,0.21,0.999957,-17.0731,2.355e-65,True
1,A vs C,Purchase rate,0.0499995,0.0599997,1.00003,-30.2357,8.035e-201,True
2,A vs B,CTR,0.2,0.16,-3.99995,71.7989,< 1e-300,True
3,A vs B,Purchase rate,0.0499995,0.0999999,5.00004,-130.912,< 1e-300,True


### Вывод по тестам долей

**A/A: A против C**

Для `CTR` и `purchase rate` p-value меньше `0.01`, поэтому отвергаем гипотезу о равенстве долей. Это означает, что A/A-группы статистически значимо отличаются.

**A/B: A против B**

Для `CTR` и `purchase rate` p-value также меньше `0.01`. В группе `B` CTR ниже, а purchase rate выше. Но интерпретировать этот результат как надёжный эффект алгоритма нельзя без оговорки, потому что A/A-проверка уже показала проблему со сплитами.

## 4. Проверка GMV

GMV — денежная числовая метрика, а не доля. Поэтому для неё:

1. агрегируем GMV на уровне пользователя/сессии;
2. проверяем нормальность распределения с помощью теста Шапиро-Уилка на `alpha = 0.01`;
3. сравниваем группы по пользовательскому GMV.

In [6]:
shapiro_rows = []

for group_name in ["A", "B", "C"]:
    values = user_metrics[group_name]["gmv"].to_numpy()
    w_stat, p_value = stats.shapiro(values)

    shapiro_rows.append({
        "group": group_name,
        "metric": "gmv_per_user",
        "n_users": len(values),
        "W_stat": w_stat,
        "p_value": p_value,
        "p_value_fmt": format_p_value(p_value),
        "normal_at_alpha_0.01": p_value >= ALPHA_SHAPIRO,
    })

shapiro_df = pd.DataFrame(shapiro_rows)
shapiro_df


,group,metric,n_users,W_stat,p_value,p_value_fmt,normal_at_alpha_0.01
0,A,gmv_per_user,996,0.997468,0.125157,1.252e-01,True
1,B,gmv_per_user,996,0.998945,0.84577,8.458e-01,True
2,C,gmv_per_user,994,0.99634,0.0198226,1.982e-02,True


In [7]:
def gmv_test(group_1: str, group_2: str) -> dict:
    x = user_metrics[group_1]["gmv"].to_numpy()
    y = user_metrics[group_2]["gmv"].to_numpy()

    welch_stat, welch_p = stats.ttest_ind(x, y, equal_var=False, alternative="two-sided")
    mw_stat, mw_p = stats.mannwhitneyu(x, y, alternative="two-sided")

    return {
        "comparison": f"{group_1} vs {group_2}",
        "metric": "gmv_per_user",
        "group_1": group_1,
        "group_2": group_2,
        "group_1_mean": x.mean(),
        "group_2_mean": y.mean(),
        "diff_group2_minus_group1": y.mean() - x.mean(),
        "welch_t_stat": welch_stat,
        "welch_p_value": welch_p,
        "welch_p_value_fmt": format_p_value(welch_p),
        "welch_reject_h0_alpha_0.01": welch_p < ALPHA,
        "mannwhitney_u_stat": mw_stat,
        "mannwhitney_p_value": mw_p,
        "mannwhitney_p_value_fmt": format_p_value(mw_p),
        "mannwhitney_reject_h0_alpha_0.01": mw_p < ALPHA,
    }

gmv_tests_df = pd.DataFrame([
    gmv_test("A", "C"),
    gmv_test("A", "B"),
])

gmv_tests_df[[
    "comparison", "metric", "group_1_mean", "group_2_mean",
    "diff_group2_minus_group1", "welch_p_value_fmt",
    "welch_reject_h0_alpha_0.01", "mannwhitney_p_value_fmt",
    "mannwhitney_reject_h0_alpha_0.01",
]]

,comparison,metric,group_1_mean,group_2_mean,diff_group2_minus_group1,welch_p_value_fmt,welch_reject_h0_alpha_0.01,mannwhitney_p_value_fmt,mannwhitney_reject_h0_alpha_0.01
0,A vs C,gmv_per_user,"53,273.3","63,998.9","10,725.6",3.377e-131,True,1.104e-118,True
1,A vs B,gmv_per_user,"53,273.3","106,473","53,200",< 1e-300,True,< 1e-300,True


### Вывод по GMV

На уровне `alpha = 0.01` тест Шапиро-Уилка не отвергает нормальность пользовательского GMV во всех трёх группах, поэтому Welch t-test можно использовать для сравнения средних GMV на пользователя.

- `A vs C`: GMV на пользователя статистически значимо отличается. Это ещё одно подтверждение, что A/A-сплиты разъехались.
- `A vs B`: GMV на пользователя в группе `B` статистически значимо выше, чем в `A`. Но из-за проваленной A/A-проверки этот результат нельзя считать полностью надёжным.

## 5. Финальный вывод

1. Дублей нет, кейсов «клик/покупка без просмотра» нет.
2. Метрики рассчитаны по всем датасетам.
3. A/A-тест `A` против `C` не проходит:
   - `CTR` различается статистически значимо;
   - `purchase rate` различается статистически значимо;
   - `GMV на пользователя` различается статистически значимо.
4. Значит, есть проблема с разъезжанием сплитов или с дизайном/логированием эксперимента.
5. A/B-сравнение `A` против `B` показывает, что алгоритм `B`:
   - снижает CTR;
   - повышает purchase rate;
   - сильно повышает GMV на пользователя.
6. Однако на результат A/B-теста нельзя полностью положиться, потому что контрольные A/A-группы уже статистически значимо отличаются.

**Итог:** алгоритм `B` выглядит бизнесово лучше по покупкам и GMV, но эксперимент нельзя признать корректным без перепроверки сплитования. Перед принятием продуктового решения нужно исправить проблему с A/A-сплитами и перезапустить/перепроверить эксперимент.